# 3. Portfolio Construction and Risk Adjustment

## Overview
This notebook implements a sophisticated portfolio construction process that considers multiple factors for optimal asset allocation:

1. **Initial Asset Class Allocation**
   - Strategic allocation between equities (90%) and bonds (10%)
   - Regional diversification across developed and emerging markets

2. **Sharpe Ratio Adjustments**
   - Adjustment of weights based on risk-adjusted returns
   - Different sensitivity factors for equities and bonds
   - Regional category weights balanced by performance

3. **Final Portfolio Construction**
   - Risk-adjusted position sizing
   - Volatility-based cash weight adjustments
   - Final investment allocation in GBP

The process aims to create a well-diversified portfolio that balances return potential with risk management.

## Setup and Data Loading

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import load_intermediate, save_output
from etf_utils.metrics import calculate_annualized_volatility, interpolate_adjustment_factor

provider = DataProvider()

# Load screened ETFs from previous notebook
etfs_df = pd.read_csv(DATA_INTERMEDIATE / 'summary_all.csv')

## Risk Adjustment Framework

### Sharpe Ratio Adjustment System
We implement a dynamic weight adjustment system based on Sharpe ratios:

1. **Base Factors**: Scale from 0.6 (poor) to 1.48 (excellent)
2. **Relative Performance**: Adjustments based on deviation from median
3. **Asset Class Sensitivity**:
   - Equities: Â±0.1 range (more aggressive)
   - Bonds: Â±0.25 range (more conservative)

This allows the portfolio to:
- Reward strong risk-adjusted performance
- Reduce exposure to high-risk/low-return assets
- Apply appropriate sensitivity by asset class

In [2]:
# Sharpe ratio adjustment factors
trailing_sr_adjustment_factors = {
    -1.0: 0.60,
    -0.8: 0.66,
    -0.6: 0.77,
    -0.4: 0.85,
    -0.2: 0.94,
     0.0: 1.00,
     0.2: 1.11,
     0.4: 1.19,
     0.6: 1.30,
     0.8: 1.37,
     1.0: 1.48
}

# Initial risk weights
eq_risk_weight = 90  # 90% Equities
bnd_risk_weight = 10  # 10% Bonds


# Create regional risk weights DataFrame
regional_risk_weights_data = {
    'category': [
        'Developed_AmericasandUK',
        'Developed_EMEA',
        'Developed_APAC',
        'Emerging_Americas',
        'Emerging_APACandEMEA'
    ],
    'allocation': [10, 35, 35, 10, 10],
}
  
regional_risk_weights = pd.DataFrame(regional_risk_weights_data)

In [3]:
etfs_df.columns

Index(['wkn', 'ticker', 'valor', 'name', 'inception_date', 'age_in_days',
       'age_in_years', 'strategy', 'domicile_country', 'currency', 'hedged',
       'securities_lending', 'dividends', 'ter', 'replication', 'size',
       'is_sustainable', 'number_of_holdings', 'yesterday', 'last_week',
       'last_month', 'last_three_months', 'last_six_months', 'last_year',
       'last_three_years', 'last_five_years', '2024', '2023', '2022', '2021',
       'last_dividends', 'last_year_dividends', 'last_year_volatility',
       'last_three_years_volatility', 'last_five_years_volatility',
       'last_year_return_per_risk', 'last_three_years_return_per_risk',
       'last_five_years_return_per_risk', 'max_drawdown',
       'last_year_max_drawdown', 'last_three_years_max_drawdown',
       'last_five_years_max_drawdown', 'region', 'country', 'region_category',
       'asset_class', 'intra_region_category', 'benchmark_ticker',
       'benchmark_description', 'benchmark_2024_Return', 'beta',
     

## Performance Data Collection

The following code block retrieves and calculates key performance metrics for benchmark ETFs:

- Total returns for 2024 YTD
- Annualized volatility
- Sharpe ratios for benchmarks

These metrics form the basis for our risk adjustment calculations.

In [ ]:
# Calculate benchmark returns and volatility for 2024 (same as notebook 02)
benchmark_year_start = "2024-01-01"
benchmark_year_end = "2024-12-31"

eq_prices = provider.get_historical_prices("VEVE")
bnd_prices = provider.get_historical_prices("SAAA")

eq_year = eq_prices.loc[benchmark_year_start:benchmark_year_end]["close"]
bnd_year = bnd_prices.loc[benchmark_year_start:benchmark_year_end]["close"]

eq_benchmark_return = round((eq_year.iloc[-1] - eq_year.iloc[0]) / eq_year.iloc[0] * 100, 2)
bnd_benchmark_return = round((bnd_year.iloc[-1] - bnd_year.iloc[0]) / bnd_year.iloc[0] * 100, 2)
eq_benchmark_volatility = round(calculate_annualized_volatility(eq_year) * 100, 2)
bnd_benchmark_volatility = round(calculate_annualized_volatility(bnd_year) * 100, 2)

eq_sharpe_ratio = round(eq_benchmark_return / eq_benchmark_volatility, 2)
bond_sharpe_ratio = round(bnd_benchmark_return / bnd_benchmark_volatility, 2)

print(f"2024 VEVE return: {eq_benchmark_return}%, volatility: {eq_benchmark_volatility}%, Sharpe: {eq_sharpe_ratio}")
print(f"2024 SAAA return: {bnd_benchmark_return}%, volatility: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")

## Weight Interpolation Function

This helper function implements the core logic for Sharpe ratio adjustments:

1. Takes a Sharpe ratio as input
2. Maps it to the appropriate adjustment factor
3. Handles edge cases (above/below range)
4. Returns interpolated weight adjustment

The function ensures smooth transitions between adjustment levels.

In [ ]:
# interpolate_adjustment_factor is imported from etf_utils.metrics
# Example usage:
# factor = interpolate_adjustment_factor(sharpe_ratio, trailing_sr_adjustment_factors)

## Final Portfolio Construction

The following calculations implement the complete portfolio construction process:

1. **Risk Weight Calculation**
   - Apply Sharpe ratio adjustments
   - Normalize weights within asset classes

2. **Cash Weight Determination**
   - Convert risk weights to cash weights
   - Account for volatility differences

3. **Regional Balance**
   - Maintain target regional exposures
   - Adjust for relative strength within regions

4. **Final Allocation**
   - Calculate GBP investment amounts
   - Ensure proper diversification
   - Generate actionable trade list

The result is a complete portfolio allocation with specific investment amounts for each ETF.

In [10]:
# Get adjustment factors
eq_adj_factor = interpolate_adjustment_factor(eq_sharpe_ratio, trailing_sr_adjustment_factors)
bond_adj_factor = interpolate_adjustment_factor(bond_sharpe_ratio, trailing_sr_adjustment_factors)

# Calculate adjusted risk weights
eq_adj_risk_weights = eq_risk_weight * eq_adj_factor
bond_adj_risk_weights = bnd_risk_weight * bond_adj_factor

# Normalize weights and ensure total is 100%
total_adj_weight = eq_adj_risk_weights + bond_adj_risk_weights
normalized_eq_weight = round((eq_adj_risk_weights / total_adj_weight)*100,2)
normalized_bond_weight = round((bond_adj_risk_weights / total_adj_weight)*100,2)
total_normalized_weight = normalized_eq_weight + normalized_bond_weight

# Calculate volatility-adjusted cash weights in percentage
eq_cash_weights = normalized_eq_weight / eq_benchmark_volatility
bond_cash_weights = normalized_bond_weight / bnd_benchmark_volatility

# Final cash weights
total_cash_weight = eq_cash_weights + bond_cash_weights
final_eq_cash_weight = total_normalized_weight*eq_cash_weights / total_cash_weight
final_bond_cash_weight = total_normalized_weight*bond_cash_weights / total_cash_weight

print(f'Asset Weights:')
print(f'Equities: {final_eq_cash_weight:.2f}')
print(f'Bonds: {final_bond_cash_weight:.2f}')

Asset Weights:
Equities: 42.91
Bonds: 57.09


In [15]:
#For etfs_df, for each region category, divide 100 equally between the different intra region categories, round it to a whole number and assign it to the intra_region_category_weight column
etfs = etfs_df
etfs['region_category_weight'] = etfs['region_category'].map(regional_risk_weights.set_index('category')['allocation'])
etfs['intra_region_category_weight'] = 0

for asset in etfs['asset_class'].unique():
    for region_category in etfs[etfs['asset_class']==asset]['region_category'].unique():
        intra_region_categories = etfs[(etfs['asset_class'] == asset) & (etfs['region_category'] == region_category)]['intra_region_category'].unique()
        num_categories = len(intra_region_categories)
        if num_categories > 0:
            weight_per_category = 100 // num_categories
            etfs.loc[etfs['region_category'] == region_category, 'intra_region_category_weight'] = weight_per_category

# Intra asset Sharpe ratio adjustment factors
intra_asset_bond_trailing_sr_data = {
    -0.25: 0.6,
    -0.2: 0.66,
    -0.15: 0.77,
    -0.1: 0.85,
    -0.05: 0.94,
    0: 1,
    0.05: 1.11,
    0.1: 1.19,
    0.15: 1.3,
    0.2: 1.37,
    0.25: 1.48
}

intra_asset_equity_trailing_sr_data = {
    -0.1: 0.6,
    -0.08: 0.66,
    -0.06: 0.77,
    -0.04: 0.85,
    -0.02: 0.94,
    0: 1,
    0.02: 1.11,
    0.04: 1.19,
    0.06: 1.3,
    0.08: 1.37,
    0.1: 1.48
}

etfs['starting_risk_weights'] = etfs['region_category_weight'] * etfs['intra_region_category_weight'] / 100
etfs['yield'] = (etfs['last_year_dividends'] - etfs['ter']).round(2)
etfs['sharpe_ratio'] = (etfs['yield'] / etfs['last_year_volatility']).round(2)
median_sharpe_ratio_equities = etfs[etfs['asset_class'] == 'equity']['sharpe_ratio'].median()
median_sharpe_ratio_bonds = etfs[etfs['asset_class'] == 'bonds']['sharpe_ratio'].median()
etfs['relative_sharpe_ratio'] = etfs.apply(lambda row: (row['sharpe_ratio'] - median_sharpe_ratio_equities) if row['asset_class'] == 'equity' else (row['sharpe_ratio'] - median_sharpe_ratio_bonds), axis=1)
etfs['interpolated_adjusted_factors'] = etfs.apply(lambda row: interpolate_adjustment_factor(row['relative_sharpe_ratio'],intra_asset_equity_trailing_sr_data) if row['asset_class'] == 'equity' else interpolate_adjustment_factor(row['relative_sharpe_ratio'], intra_asset_bond_trailing_sr_data), axis=1)
etfs['adjusted_risk_weights'] = (etfs.apply(lambda row: row['interpolated_adjusted_factors'] * row['starting_risk_weights'], axis=1)).round(2)
#normalize the adjusted risk weights to sum to 100% for each asset class
etfs['normalized_risk_weights'] = etfs.groupby('asset_class')['adjusted_risk_weights'].transform(lambda x: x / x.sum() * 100)
#calculate cash weights guess as normalized risk weights divided by volatility
etfs['cash_weights_guess'] = etfs['normalized_risk_weights'] / etfs['last_year_volatility']
#For each asset class, calculate cash weight for each etf as ((sum of normalized risk weights) / sum of all cash weights guess ) * (cash weights guess for that etf)
etfs['cash_weights'] = (etfs.groupby('asset_class').apply(lambda x: (x['normalized_risk_weights'].sum() / x['cash_weights_guess'].sum()) * x['cash_weights_guess']).reset_index(drop=True)).round(2)
# calculate final regional category weights as the sum of the cash weights per each region category per asset class
etfs['region_category_weight_final'] = etfs.groupby(['asset_class', 'region_category'])['cash_weights'].transform('sum')
# Per asset class, add normalized asset class weights to the dataframe : normalized_eq_weight and normalized_bond_weight
etfs['normalized_asset_class_weight'] = etfs['asset_class'].apply(lambda x: normalized_eq_weight if x == 'equity' else normalized_bond_weight)

#calculate final risk weight as normalized asset class weight  * normalized risk weights
etfs['final_risk_weights'] = etfs['normalized_asset_class_weight'] * etfs['normalized_risk_weights'] / 100

#calculate final cash weights guess as final risk weights divided by volatility
etfs['final_cash_weights_guess'] = etfs['final_risk_weights'] / etfs['last_year_volatility']

# For all asset classes together, calculate final cash weights for each etf as ((sum of final_risk_weights) / sum of all final_cash_weights_guess ) * (final_cash_weights_guess for that etf)
etfs['final_cash_weights'] = etfs['final_risk_weights'].sum() / etfs['final_cash_weights_guess'].sum() * etfs['final_cash_weights_guess']
etfs['final_cash_weights'] = etfs['final_cash_weights'].round()

# Invest 40000 GBP into each ETF as per final cash weights
etfs['investment'] = 20000 * etfs['final_cash_weights'] / 100

etfs.to_csv(DATA_OUTPUT / 'final_portfolio.csv', index=False)

#sort the etfs by investment in descending order 
etfs.sort_values(by=['asset_class','investment'], ascending=[False,False])[['asset_class',
      'region_category',
      'country',
      'intra_region_category',
      'ticker',
      'name',
      'final_cash_weights',
      'beta',
      'yday_close_price',
      'ter',
      'investment',
      'available_on_investengine']]

C:\Users\rakes\AppData\Local\Temp\ipykernel_64768\832020005.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  etfs['cash_weights'] = (etfs.groupby('asset_class').apply(lambda x: (x['normalized_risk_weights'].sum() / x['cash_weights_guess'].sum()) * x['cash_weights_guess']).reset_index(drop=True)).round(2)


,asset_class,region_category,country,intra_region_category,ticker,name,final_cash_weights,beta,yday_close_price,ter,investment,available_on_investengine
4,equity,Developed_EMEA,Italy,High Yield,IMIB,iShares FTSE MIB UCITS ETF EUR (Dist),19.0,NaN,2012.75,0.35,3800.0,True
5,equity,Developed_EMEA,Germany,Beta,XDDX,Xtrackers DAX ESG Screened UCITS ETF 1D,17.0,1.606825,12650.00,0.09,3400.0,True
0,equity,Developed_APAC,Australia,High Yield,AUAD,UBS ETF (IE) MSCI Australia UCITS ETF (AUD) A-dis,16.0,NaN,1827.00,0.40,3200.0,True
3,equity,Developed_AmericasandUK,United Kingdom,Beta,LCUK,Amundi UK Equity All Cap UCITS ETF Dist,10.0,1.030462,12.36,0.04,2000.0,True
7,equity,Emerging_Americas,Brazil,High Yield,IBZL,iShares MSCI Brazil UCITS ETF (Dist),9.0,NaN,1640.00,0.74,1800.0,True
1,equity,Developed_APAC,Japan,Beta,PRIJ,Amundi Prime Japan UCITS ETF DR (D),7.0,1.092259,2368.25,0.05,1400.0,True
2,equity,Developed_AmericasandUK,United States,High Yield,QYLP,Global X Nasdaq 100 Covered Call UCITS ETF D,7.0,NaN,11.45,0.45,1400.0,True
6,equity,Emerging_APACandEMEA,China,High Yield,HMCH,HSBC MSCI China UCITS ETF USD,2.0,NaN,554.62,0.28,400.0,True
15,bonds,Developed_EMEA,Germany,Corp,VECP,Vanguard EUR Corporate Bond UCITS ETF Distribu...,5.0,NaN,40.88,0.09,1000.0,True
9,bonds,Developed_AmericasandUK,United Kingdom,Corp,SLXX,iShares Core GBP Corporate Bond UCITS ETF,2.0,NaN,120.74,0.20,400.0,True


In [16]:
etfs[etfs['available_on_investengine']==True][['asset_class',
      'region_category',
      'country',
      'intra_region_category',
      'ticker',
      'name',
      'final_cash_weights',
      'beta',
      'yday_close_price',
      'ter',
      'investment',
      'available_on_investengine']]

,asset_class,region_category,country,intra_region_category,ticker,name,final_cash_weights,beta,yday_close_price,ter,investment,available_on_investengine
0,equity,Developed_APAC,Australia,High Yield,AUAD,UBS ETF (IE) MSCI Australia UCITS ETF (AUD) A-dis,16.0,NaN,1827.00,0.40,3200.0,True
1,equity,Developed_APAC,Japan,Beta,PRIJ,Amundi Prime Japan UCITS ETF DR (D),7.0,1.092259,2368.25,0.05,1400.0,True
2,equity,Developed_AmericasandUK,United States,High Yield,QYLP,Global X Nasdaq 100 Covered Call UCITS ETF D,7.0,NaN,11.45,0.45,1400.0,True
3,equity,Developed_AmericasandUK,United Kingdom,Beta,LCUK,Amundi UK Equity All Cap UCITS ETF Dist,10.0,1.030462,12.36,0.04,2000.0,True
4,equity,Developed_EMEA,Italy,High Yield,IMIB,iShares FTSE MIB UCITS ETF EUR (Dist),19.0,NaN,2012.75,0.35,3800.0,True
5,equity,Developed_EMEA,Germany,Beta,XDDX,Xtrackers DAX ESG Screened UCITS ETF 1D,17.0,1.606825,12650.00,0.09,3400.0,True
6,equity,Emerging_APACandEMEA,China,High Yield,HMCH,HSBC MSCI China UCITS ETF USD,2.0,NaN,554.62,0.28,400.0,True
7,equity,Emerging_Americas,Brazil,High Yield,IBZL,iShares MSCI Brazil UCITS ETF (Dist),9.0,NaN,1640.00,0.74,1800.0,True
8,bonds,Developed_AmericasandUK,United Kingdom,Govt,IGLT,iShares Core UK Gilts UCITS ETF,0.0,NaN,9.70,0.07,0.0,True
9,bonds,Developed_AmericasandUK,United Kingdom,Corp,SLXX,iShares Core GBP Corporate Bond UCITS ETF,2.0,NaN,120.74,0.20,400.0,True


In [17]:
#group weights by asset class and region category
etfs.groupby(['region_category'])[['final_cash_weights']].sum().sort_values(by='final_cash_weights', ascending=False).reset_index()

,region_category,final_cash_weights
0,Developed_EMEA,43.0
1,Developed_APAC,23.0
2,Developed_AmericasandUK,21.0
3,Emerging_Americas,9.0
4,Emerging_APACandEMEA,4.0


In [60]:
regional_risk_weights_data

{'category': ['Developed_AmericasandUK',
  'Developed_EMEA',
  'Developed_APAC',
  'Emerging_Americas',
  'Emerging_APACandEMEA'],
 'allocation': [10, 35, 35, 10, 10]}